# 06. End-to-End Demo

Collect the benchmark outputs into one research-summary figure and a compact run summary table.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

FIGURE_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR = ROOT / 'outputs' / 'tables'
LABEL_DIR = ROOT / 'data' / 'labels'
SCAN_DIR = ROOT / 'data' / 'scanned_pointclouds'

required = [
    FIGURE_DIR / 'reference_synthetic_rockpile_and_psd.png',
    FIGURE_DIR / 'synthetic_rockpile_exterior_scan_preview.png',
    FIGURE_DIR / 'synthetic_rockpile_exterior_dbscan_segmentation.png',
    FIGURE_DIR / 'synthetic_rockpile_estimated_vs_ground_truth_psd.png',
    FIGURE_DIR / 'p80_sensitivity_heatmap.png',
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Run notebooks 01-05 first. Missing: ' + ', '.join(str(p) for p in missing))
print('All required outputs found.')

In [ ]:
gt_summary = pd.read_csv(TABLE_DIR / 'ground_truth_summary.csv').iloc[0]
seg_metrics = pd.read_csv(TABLE_DIR / 'segmentation_metrics.csv').iloc[0]
p80_comparison = pd.read_csv(TABLE_DIR / 'p80_comparison.csv').iloc[0]
sensitivity = pd.read_csv(TABLE_DIR / 'p80_error_sensitivity.csv')

run_summary = pd.DataFrame([{
    'n_ground_truth_fragments': int(gt_summary['n_fragments']),
    'ground_truth_P80_mm': float(gt_summary['P80_mm']),
    'dbscan_ARI': float(seg_metrics['adjusted_rand_index']),
    'dbscan_clusters': int(seg_metrics['n_predicted_clusters']),
    'estimated_P80_mm': float(p80_comparison['estimated_P80_mm']),
    'P80_relative_error_pct': float(p80_comparison['P80_relative_error_pct']),
    'mean_sensitivity_P80_error_pct': float(sensitivity['P80_relative_error_pct'].mean()),
}])
summary_path = TABLE_DIR / 'end_to_end_run_summary.csv'
run_summary.to_csv(summary_path, index=False)
run_summary

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()
panels = [
    ('Synthetic_Rockpile reference and PSD', FIGURE_DIR / 'reference_synthetic_rockpile_and_psd.png'),
    ('Exterior-only point cloud scan', FIGURE_DIR / 'synthetic_rockpile_exterior_scan_preview.png'),
    ('DBSCAN on exterior scan', FIGURE_DIR / 'synthetic_rockpile_exterior_dbscan_segmentation.png'),
    ('Estimated vs Synthetic_Rockpile PSD', FIGURE_DIR / 'synthetic_rockpile_estimated_vs_ground_truth_psd.png'),
    ('P80 sensitivity heatmap', FIGURE_DIR / 'p80_sensitivity_heatmap.png'),
]
for ax, (title, path) in zip(axes, panels):
    ax.imshow(mpimg.imread(path))
    ax.set_title(title, fontsize=10)
    ax.axis('off')

axes[-1].axis('off')
text = (
    f"Fragments: {int(run_summary.loc[0, 'n_ground_truth_fragments'])}\n"
    f"GT P80: {run_summary.loc[0, 'ground_truth_P80_mm']:.1f} mm\n"
    f"DBSCAN ARI: {run_summary.loc[0, 'dbscan_ARI']:.3f}\n"
    f"Estimated P80: {run_summary.loc[0, 'estimated_P80_mm']:.1f} mm\n"
    f"P80 error: {run_summary.loc[0, 'P80_relative_error_pct']:.1f}%"
)
axes[-1].text(0.02, 0.90, text, va='top', ha='left', fontsize=13, family='monospace')
axes[-1].set_title('Run summary', fontsize=10)

fig.suptitle('Synthetic-to-Real 3D Rock Fragmentation Benchmark', fontsize=16)
fig.tight_layout()
figure_path = FIGURE_DIR / 'end_to_end_benchmark_summary.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print(figure_path)